In [1]:
import pandas as pd
import sqlite3

df = pd.read_csv('netflix_titles.csv')
conn = sqlite3.connect('netflix.db')
df.to_sql('netflix', conn, if_exists='replace', index=False)

print("Loaded", len(df), "rows")

Loaded 8807 rows


In [2]:
query = "SELECT * FROM netflix LIMIT 5"
pd.read_sql(query, conn)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,None,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,None,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",None,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,None,None,None,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,None,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [3]:
query = """
SELECT 
    SUM(CASE WHEN director IS NULL OR director = '' THEN 1 ELSE 0 END) as missing_director,
    SUM(CASE WHEN country IS NULL OR country = '' THEN 1 ELSE 0 END) as missing_country,
    SUM(CASE WHEN rating IS NULL OR rating = '' THEN 1 ELSE 0 END) as missing_rating
FROM netflix
"""
pd.read_sql(query, conn)

,missing_director,missing_country,missing_rating
0,2634,831,4


In [ ]:
Data Quality):Out of ~8,807 total rows, 2,634 rows (~30%) are 
missing director information, 831 rows (~9%) are missing country, and only 4 rows are 
missing rating. Director is the weakest field — likely because many titles (documentaries, 
stand-up specials, some international content) don't always credit a single director. 
This means any "top director" analysis only covers ~70% of the catalog, and country-based 
analysis excludes ~9% of titles.

In [6]:
query = """
SELECT type, COUNT(*) as total
FROM netflix
GROUP BY type
"""
pd.read_sql(query, conn)

,type,total
0,Movie,6131
1,TV Show,2676


Netflix's catalog is 6,131 Movies (69.6%) vs 2,676 TV Shows (30.4%) 
— Movies dominate the platform roughly 2:1 over TV Shows.

In [5]:
query = """
SELECT country, COUNT(*) as total
FROM netflix
WHERE country IS NOT NULL AND country != ''
GROUP BY country
ORDER BY total DESC
LIMIT 10
"""
pd.read_sql(query, conn)

,country,total
0,United States,2818
1,India,972
2,United Kingdom,419
3,Japan,245
4,South Korea,199
5,Canada,181
6,Spain,145
7,France,124
8,Mexico,110
9,Egypt,106


#India is the 2nd largest content producer (after US)

In [7]:
query = """
SELECT type, rating, COUNT(*) as total
FROM netflix
WHERE rating IS NOT NULL AND rating != ''
GROUP BY type, rating
ORDER BY type, total DESC
"""
pd.read_sql(query, conn)

,type,rating,total
0,Movie,TV-MA,2062
1,Movie,TV-14,1427
2,Movie,R,797
3,Movie,TV-PG,540
4,Movie,PG-13,490
5,Movie,PG,287
6,Movie,TV-Y7,139
7,Movie,TV-Y,131
8,Movie,TV-G,126
9,Movie,NR,75


In [8]:
query = """
SELECT COUNT(*) as bad_rating_rows
FROM netflix
WHERE rating LIKE '%min%'
"""
pd.read_sql(query, conn)

,bad_rating_rows
0,3


 TV-MA is the most common rating for both Movies (2,062 titles) 
and TV Shows (1,145 titles), showing Netflix's catalog skews toward mature/adult audiences 
rather than family content. Also identified a data quality issue: 3 rows have "min" values 
incorrectly stored in the rating column (e.g., "84 min") instead of an actual rating — 
these will be excluded or cleaned before final analysis.

In [9]:
query = """
SELECT type,
       AVG(CAST(REPLACE(duration, ' min', '') AS INTEGER)) as avg_duration_minutes
FROM netflix
WHERE type = 'Movie' AND duration LIKE '%min%'
"""
pd.read_sql(query, conn)

,type,avg_duration_minutes
0,Movie,99.577187


In [10]:
query = """
SELECT type,
       AVG(CAST(REPLACE(REPLACE(duration, ' Seasons', ''), ' Season', '') AS INTEGER)) as avg_seasons
FROM netflix
WHERE type = 'TV Show'
"""
pd.read_sql(query, conn)

,type,avg_seasons
0,TV Show,1.764948


Average movie duration is ~99.6 minutes (about 1h 40m). 
Average TV Show length is ~1.76 seasons, meaning most Netflix shows don't run long — 
the majority are 1-2 seasons only.

In [11]:
query = """
SELECT director, COUNT(*) as total_titles
FROM netflix
WHERE director IS NOT NULL AND director != ''
GROUP BY director
ORDER BY total_titles DESC
LIMIT 10
"""
pd.read_sql(query, conn)

,director,total_titles
0,Rajiv Chilaka,19
1,"Raúl Campos, Jan Suter",18
2,Suhas Kadav,16
3,Marcus Raboy,16
4,Jay Karas,14
5,Cathy Garcia-Molina,13
6,Youssef Chahine,12
7,Martin Scorsese,12
8,Jay Chapman,12
9,Steven Spielberg,11


 Rajiv Chilaka leads with 19 titles, followed by the duo 
Raúl Campos & Jan Suter (18 titles) and Suhas Kadav (16). Notably, several top directors 
(Chilaka, Kadav) specialize in Indian animated/children's content — suggesting Netflix 
has invested heavily in Indian-produced kids' content specifically, not just adult drama.

In [12]:
query = """
WITH yearly_totals AS (
    SELECT release_year, COUNT(*) as total
    FROM netflix
    GROUP BY release_year
)
SELECT * FROM yearly_totals
ORDER BY release_year DESC
LIMIT 10
"""
pd.read_sql(query, conn)

,release_year,total
0,2021,592
1,2020,953
2,2019,1030
3,2018,1147
4,2017,1032
5,2016,902
6,2015,560
7,2014,352
8,2013,288
9,2012,237


Content additions grew steadily from 2012 (237 titles) to a peak 
in 2018 (1,147 titles), then began declining — 2019: 1,030, 2020: 953, 2021: 592. 
Note: 2021 figure is likely incomplete since this dataset was captured mid-year, 
not a genuine decline.

In [13]:
query = """
WITH genre_yearly AS (
    SELECT release_year, listed_in as genre, COUNT(*) as total
    FROM netflix
    WHERE release_year >= 2019
    GROUP BY release_year, listed_in
)
SELECT * FROM genre_yearly
WHERE genre LIKE '%Documentar%' OR genre LIKE '%Stand-Up%'
ORDER BY release_year
"""
pd.read_sql(query, conn)

,release_year,genre,total
0,2019,"Children & Family Movies, Documentaries, Sport...",1
1,2019,Documentaries,33
2,2019,"Documentaries, Dramas",1
3,2019,"Documentaries, Faith & Spirituality",1
4,2019,"Documentaries, International Movies",27
5,2019,"Documentaries, International Movies, Music & M...",4
6,2019,"Documentaries, International Movies, Sports Mo...",6
7,2019,"Documentaries, LGBTQ Movies",2
8,2019,"Documentaries, Music & Musicals",21
9,2019,"Documentaries, Sports Movies",7


Contrary to the assumption that Documentaries/Stand-Up Comedy 
grew ~40% recently, the actual data shows a decline in pure "Documentaries" titles 
(33 → 23 → 20 from 2019-2021) and "Stand-Up Comedy" (48 → 41 → 12). This may partly 
reflect incomplete 2021 data (dataset cutoff), but the trend does not support a 
"fast-growing genre" claim as-is — worth flagging as a case where assumption ≠ data.

In [14]:
query = """
WITH genre_counts AS (
    SELECT release_year, listed_in as genre, COUNT(*) as total
    FROM netflix
    GROUP BY release_year, listed_in
)
SELECT release_year, genre, total,
       RANK() OVER (PARTITION BY release_year ORDER BY total DESC) as rank
FROM genre_counts
"""
result = pd.read_sql(query, conn)
result[result['rank'] == 1].sort_values('release_year', ascending=False).head(10)

,release_year,genre,total,rank
2641,2021,Kids' TV,24,1
2428,2020,"Dramas, International Movies",42,1
2203,2019,Stand-Up Comedy,48,1
1958,2018,Stand-Up Comedy,58,1
1760,2017,Documentaries,89,1
1569,2016,Documentaries,67,1
1410,2015,"Dramas, International Movies",30,1
1293,2014,"Comedies, Dramas, International Movies",16,1
1190,2013,Children & Family Movies,19,1
1100,2012,Stand-Up Comedy,12,1


The #1 genre by year shifted over time — Stand-Up Comedy led in 
2012, 2018, and 2019; Documentaries led in 2016-2017; Dramas/International Movies led in 
2014-2015 and 2020. This shows Netflix's genre focus has fluctuated rather than trending 
consistently toward one category — no single genre has dominated every year.

While Documentaries and Stand-Up Comedy were strong genres in 2018-2019, recent data shows declining additions in these categories (accounting for incomplete 2021 data) — rather than assuming continued growth, Netflix should verify current-year content strategy against this trend before further genre-specific investment.

In [2]:
# Main cleaned table
df_clean = pd.read_sql("SELECT * FROM netflix", conn)
df_clean.to_csv('netflix_cleaned.csv', index=False)

# Country breakdown
country_data = pd.read_sql("""
    SELECT country, COUNT(*) as total 
    FROM netflix 
    WHERE country IS NOT NULL AND country != '' 
    GROUP BY country 
    ORDER BY total DESC
""", conn)
country_data.to_csv('country_breakdown.csv', index=False)

# Genre trend by year
genre_trend = pd.read_sql("""
    SELECT release_year, listed_in as genre, COUNT(*) as total
    FROM netflix
    WHERE release_year >= 2015
    GROUP BY release_year, listed_in
""", conn)
genre_trend.to_csv('genre_trend.csv', index=False)

rating_data = pd.read_sql("""
    SELECT type, rating, COUNT(*) as total
    FROM netflix
    WHERE rating IS NOT NULL AND rating != ''
    GROUP BY type, rating
    ORDER BY type, total DESC
""", conn)
rating_data.to_csv('rating_by_type.csv', index=False)


print("All files exported")

All files exported
